# Expense Claim Analysis

Dis notebook show how to create agents wey dey use plugins to process travel expenses from local receipt images, generate expense claim email, and visualize expense data using pie chart. Agents go dynamically choose functions based on the task context.

Steps:
1. OCR Agent go process the local receipt image and extract travel expense data.
2. Email Agent go generate expense claim email.

### Example of travel expense scenario:
Imagine say you be employee wey dey travel for business meeting for another city. Your company get policy to reimburse all reasonable travel-related expenses. Na so dem dey divide potential travel expenses:
- Transportation:
Airfare for round trip from your home city go the destination city.
Taxi or ride-hailing services to and from airport.
Local transportation for destination city (like public transit, rental cars, or taxis).

- Accommodation:
Hotel stay for three nights for mid-range business hotel wey near the meeting venue.

- Meals:
Daily meal allowance for breakfast, lunch, and dinner, based on the company per diem policy.

- Miscellaneous Expenses:
Parking fees for airport.
Internet access charges for hotel.
Tips or small service charges.

- Documentation:
You go submit all receipts (flights, taxis, hotel, meals, etc.) and completed expense report for reimbursement.


## Import required libraries

Import di necessary libraries an modules for di notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Define Expense Models

 Create one Pydantic model for individual expenses and one ExpenseFormatter class wey go turn user query to structured expense data.

 Every expense go dey represent like dis:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generating the Email

Create one tool function wey go generate email for submit expense claim.
- Dis tool dey use the `@tool` decorator from di Microsoft Agent Framework.
- E dey calculate di total amount of expense dem and e dey format di details into email body.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Tool for Extracting Travel Expenses from Receipt Images

Create one tool function to comot travel expenses from receipt images.
- Dis tool dey use the `@tool` decorator from Microsoft Agent Framework.
- E go read the receipt image, encode am as base64, and return the data URI make the agent fit analyze.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processing Expenses

Define di agents dem and connect dem inside one after di oda workflow using `WorkflowBuilder`.
- Di OCR agent dey comot structured expense data from di receipt image using di `load_receipt_image` tool.
- Di Email agent go carry di data wey dem comot come create professional expense claim email using di `generate_expense_email` tool.
- `WorkflowBuilder` wit `add_edge` dey create one after oda pipeline: OCR Agent → Email Agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

Build di sequential workflow and run am to process di receipt image and generate di expense claim email.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis dokument don translate with AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even though we try make e correct, abeg sabi say automated translations fit get some mistakes or wahala. Di original dokument wey dem write for im own language na di main ogbonge source. For important matter, make you use human professional translation. We no go take responsibility for any misunderstanding or wrong meaning wey fit happen from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
